# Predicting Customer Churn at Greenweez

**Applying Classification to a Real Business Problem** 🎯

## 🏢 The Challenge

Greenweez wants to identify first-time customers at risk of **churning** (not making a second purchase within 3 months).

**Why it matters:**
- Acquiring new customers costs 5-7x more than retaining existing ones
- Limited marketing budget means we must **target the right customers**
- A small nudge (discount email) can convert "on the fence" customers

**Your task:** Build a classifier to predict repurchase probability, then design a targeted retention campaign.

## 📊 The Data

- **172,523 first-time customers** from Greenweez (2019-2021)
- **Target:** `re_purchase` (1 = repurchased, 0 = didn't)
- **Features:** First order behavior (basket size, promo usage, etc.)

## 🎯 What You'll Learn

Unlike previous classification exercises, this focuses on **business application**:

1. Interpret confusion matrices in **business terms** (wasted spend vs missed revenue)
2. Choose between **precision vs recall** based on campaign costs
3. Use **prediction probabilities** to segment customers (not just binary yes/no)
4. Design a **production workflow** for automated retention campaigns

**Key insight:** Not all customers need the same effort. Find the "persuadable" ones.

Let's dive in! 👇


## Data exploration


a) We are going to use a [table from BigQuery](https://console.cloud.google.com/bigquery?ws=!1m5!1m4!4m3!1sdata-analytics-bootcamp-363212!2scourse33!3sgwz_churn) for our business data. To access it, you need to have the right permissions. You don't have those permissions on the `data-analytics-bootcamp` project in BigQuery, so you will need to copy the table to your own project.

There are many ways to do that, the easiest is to select the `Copy` option in BigQuery and change the Destination Project and Dataset to your own project (you used this in the SQL module).

👉 **First, set your `project_id` in the cell below, then run the credential check to confirm your BigQuery setup is working correctly before loading data.**

In [ ]:
# 👇 Replace with your own GCP project ID
project_id = "your-project-id"

In [ ]:
import subprocess

result = subprocess.run(
    ['bq', 'ls', f'--project_id={project_id}'],
    capture_output=True,
    text=True
)
if result.returncode == 0:
    print("✅ BigQuery connection successful! You're ready to load data.")
else:
    print("❌ Could not connect to BigQuery. Your credentials may be expired or incorrect.")
    print("👉 Re-run the gcloud auth setup from ![setup day](https://github.com/lewagon/data-analytics-setup/tree/master) or ask a TA for help.")

👉 Second, run this cell to import the table using `pandas_gbq` and filter your `df`


In [ ]:
import pandas as pd
import pandas_gbq

dataset_id = "course33"
table_id = "gwz_churn"

# Query Bigquery
query = f"SELECT * FROM `{project_id}.{dataset_id}.{table_id}`"

df = pandas_gbq.read_gbq(query, project_id=project_id)

# The table contains multiple orders per customer.
# We keep only each customer's first order so that:
#   1. Each row represents one unique customer (no duplicate index after set_index)
#   2. There is no data leakage when we train/test split by customer
df = df.sort_values('date_date').drop_duplicates(subset='customers_id', keep='first')

print(f"✅ Loaded {len(df):,} unique first-time customers")
df.head()

b) Let's take a look at our data.

1. As usual, it's useful to first look at the first few rows.
2. What's the shape of our data?
3. Are there any null values?  
4. Given that we are trying to predict 'reconversions', what is our target?

In [ ]:
# YOUR CODE HERE

In [ ]:
# YOUR CODE HERE

In [ ]:
# YOUR CODE HERE

> YOUR ANSWER HERE

c) What do you think of column `orders_id` for our problem? Is it useful for our analysis?

> YOUR ANSWER HERE

d) Now, delete `orders_id` and `date_date` columns.

In [ ]:
# YOUR CODE HERE

e) Have a look at whether our columns values are on different scales. To do this, use the DataFrame `.describe()` method to compare them.

What kind of preprocessing we will have to use ?

In [ ]:
# YOUR CODE HERE

<details>
    <summary><i>Answer:</i></summary>

The features are indeed on different scales. For example, total_nb_codes in comparison with avg_basket.

As a result, we will need to use a normalization technique.
</details>

f) Set column `customers_id` as index to keep customer_id information.

In [ ]:
# YOUR CODE HERE

Note that for the sake of the exercise, we've **already preprocessed some of the data for you** 🔧.

This means you'll be working on a (relatively) clean database, with your targets and features already formed. In a real-world situation, it's likely that you'll be spending a lot of time forming your target and features from simpler, less-specific data, either using python or SQL to manipulate the database.

In [ ]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'data_exploration',
    df_shape=df.shape,
    df_columns=set(df.columns),
    cleaned_columns=set(df.columns),
    index_name=df.index.name
)
result.write()
print(result.check())

## Modeling

Now that we've seen what our data looks like, we need to define our target and features.

g) Split dataset into a train and a test set (this should give you an `X_train`, `X_test`, `y_train` and `y_test`).

We will keep a test_size of 20%.

In [ ]:
# YOUR CODE HERE

h) Execute the cell below to apply normalization on our data. We are going to use a StandardScaler for this transformation.

*Make sure you understand what this code does.*

Why do we use `.fit_transform()` on the train set and `.transform()` on the test set?

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)

# apply same transformation on X_test
X_test = scaler.transform(X_test)

In [ ]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'preprocessing',
    X_train_shape=X_train.shape,
    X_train_rows=X_train.shape[0],
    X_test_rows=X_test.shape[0],
    X_train_type=type(X_train).__module__ + '.' + type(X_train).__name__
)
result.write()
print(result.check())

i) What are the types of X_train and X_test?

In [ ]:
# YOUR CODE HERE

j) Before building our first model, we need a **baseline** to compare against!

**Baseline Strategy:** Predict that **all customers will churn** (always predict 0).

Calculate the baseline accuracy:

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

baseline_y_pred = np.zeros(len(y_test))

baseline_accuracy = accuracy_score(y_test, baseline_y_pred)

print(f"Baseline accuracy is {round(baseline_accuracy,2)}")

**Why a baseline?**  
This is the score to beat. If our model barely improves on predicting "everyone churns", it's not worth deploying.

**What does this accuracy tell you?**  
~71% of customers churn after their first order, so a naive "always predict churn" model already scores 71%. Our dataset is imbalanced (more 0s than 1s), which means accuracy alone will be misleading — we'll need precision and recall to evaluate whether the model is genuinely useful.

Now let's build a model that beats this baseline!


k) We will use a simple logistic regression model. Execute cell below to train your model on the train data and store the test data predictions in a variable `y_pred`.

Make sure you understand what this code does.

In [ ]:
from sklearn.linear_model import LogisticRegression

# train model
clf = LogisticRegression()
clf.fit(X_train, y_train)

# store predictions
y_pred = clf.predict(X_test)

l) Calculate the accuracy you get on test data and save it to a variable called `accuracy_test`.

How do you interpret this value?

In [ ]:
# YOUR CODE HERE

<details>
    <summary><i>Answer:</i></summary>

Logistic regression score corresponds to accuracy. It measures the proportion of predictions that are correct.

You should see around **75.6% accuracy** on the test data, with a similar score on the train set — this means the model is generalising well with no significant overfitting.

Note that the model only **narrowly beats the 71% baseline** (~5% improvement). This is actually expected — accuracy is a misleading metric here because 71% of customers churned. The model can score well simply by predicting "won't repurchase" most of the time. We'll look at precision and recall next to see whether the model is genuinely adding business value.

</details>

In [ ]:
from nbresult import ChallengeResult
from sklearn.metrics import precision_score

result = ChallengeResult(
    'model',
    baseline_accuracy=float(baseline_accuracy),
    model_accuracy=float(accuracy_test),
    precision=float(precision_score(y_test, y_pred))
)
result.write()
print(result.check())

m) Accuracy is one metric, but for business decisions we need more detail!

Run this code to see your model's confusion matrix:

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

confusion_matrix = ConfusionMatrixDisplay.from_estimator(clf, X_test, y_test,labels=[1, 0])

Let's try to make that a little easier to understand by adding some labels!

In [ ]:
import matplotlib.pyplot as plt

# Create confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_estimator(
    clf, X_test, y_test,
    labels=[0, 1],
    display_labels=["Won't Repurchase", "Will Repurchase"],
    cmap='Blues',
    ax=ax
)
plt.title('Confusion Matrix: Churn Prediction')
plt.show()

Reading the Matrix:

<img alt="Confusion Matrix Metrics" src="https://wagon-public-assets.s3.eu-west-3.amazonaws.com/pchl73lhb84nyh6ic4yzw98bl2me">

- TN (True Negative): Correctly predicted "won't repurchase"
- FP (False Positive): Predicted "will repurchase" but they didn't (wasted marketing $)
- FN (False Negative): Predicted "won't repurchase" but they did (missed opportunity)
- TP (True Positive): Correctly predicted "will repurchase"

n) Now let's calculate precision, recall, and accuracy using sklearn (don't do it manually!)

In [ ]:
from sklearn.metrics import precision_score, recall_score, accuracy_score, classification_report

# Calculate all metrics
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)

print(f"Precision: {precision:.1%}")
print(f"Recall:    {recall:.1%}")
print(f"Accuracy:  {accuracy:.1%}")

# Even better: Get a full report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Won't Repurchase", "Will Repurchase"]))

**What do these metrics mean?**

- **Precision:** Of customers we predicted would repurchase, what % actually did? (High = fewer wasted campaigns)
- **Recall:** Of customers who did repurchase, what % did we catch? (High = fewer missed opportunities)
- **Accuracy:** Overall correctness (⚠️ misleading for imbalanced data)

**For this business problem:**  
Precision matters if campaigns are expensive. Recall matters if campaigns are cheap. Greenweez should optimize for **recall** since emails cost almost nothing.

Which metric should we optimize? This depends on business priorities!

**Optimize for Precision**

- Goal: Don't waste marketing budget
- Accept: Missing some potential customers (low recall)
- Use case: Expensive retention campaigns (personalized gifts, high-value discounts)

**Optimize for Recall**

- Goal: Catch ALL potential churners
- Accept: Some false alarms (low precision)
- Use case: Cheap interventions (automated emails, small discounts)

**Balance Both**

- Use F1-score (harmonic mean of precision & recall)
- Typical for most business cases

**For Greenweez:**

Email campaigns are cheap, so optimize for recall. It's better to send a few extra emails than miss potential customers!

👉 Check your recall score: you'll see it's quite low (~27%) even though accuracy looks reasonable at ~76%. This is the **imbalanced data problem** in action — the model has learned that predicting "won't repurchase" is usually correct (71% of the time), so it rarely predicts repurchase. High precision (70%) but low recall (27%) means it's cautious: when it does predict a repurchase it's usually right, but it misses most of the real repurchasers. This is exactly why we use probability scores in the next steps instead of relying on binary predictions.

o) So far we've used the default threshold (0.5) to classify customers. But we can do better!

Run this code to see the **probabilities** behind each prediction:

In [ ]:
# Get probabilities for each class
proba = pd.DataFrame(
    clf.predict_proba(X_test),
    columns=["Won't Repurchase", "Will Repurchase"],
    index=X_test.index
)

# Sort by repurchase probability
proba_sorted = proba.sort_values("Will Repurchase", ascending=False)
proba_sorted.head(10)

**What this shows:**

- Each customer gets a probability score (0-1)
- **High probability** (>0.7): Very likely to repurchase. These are low priority for intervention
- **Medium probability** (0.2-0.5): On the fence. We should target these!
- **Low probability** (<0.2): Likely lost. Don't waste resources on these.

p) Focus retention efforts on customers with **20-50% repurchase probability** - they're on the fence and persuadable!

Filter to find these at-risk customers and save them to a variable called `at_risk_customers`

In [ ]:
# YOUR CODE HERE

**Why This Matters:**

| Probability | Customer Type | Marketing Action |
|-------------|---------------|------------------|
| < 20% | Likely lost | Don't target (low ROI) |
| 20-50% | **At-risk** | **Target with discounts** |
| 50-70% | Moderately likely | Send reminder emails |
| > 70% | Very likely | No intervention needed |

**Business Impact:**  
By targeting just the 20-50% group, you maximize ROI - these customers are persuadable but need a nudge!



In [ ]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'evaluation',
    precision=float(precision),
    recall=float(recall),
    accuracy=float(accuracy),
    proba_rows=len(proba),
    proba_cols=len(proba.columns),
    at_risk_count=len(at_risk_customers)
)
result.write()
print(result.check())

q) Now that you have a working churn prediction model, how would Greenweez use it in production?



<details>
<summary><i>One Potential Suggestion</i></summary>

**Proposed Workflow:**

1. **Weekly Batch Prediction**
   - Run model on all customers who made first purchase in past week
   - Identify at-risk customers (20-50% repurchase probability)

2. **ELT to CRM**
   - Export customer IDs + probabilities to BigQuery
   - Use dbt to transform into actionable insights
   - Sync to CRM (Salesforce, HubSpot, etc.)

3. **Automated Campaigns**
   - **Day 7:** Email with 10% discount code
   - **Day 14:** Personalized product recommendations
   - **Day 21:** Final "We miss you" campaign with 20% discount

4. **Measure & Iterate**
   - A/B test: Targeted group vs control
   - Track conversion lift
   - Retrain model monthly with new data

**Expected Results:**
- 15-25% lift in repurchase rate for at-risk segment
- ROI of 3-5x on retention campaigns
- Reduced customer acquisition costs
</details>

## 🏁 Conclusion

**🎉 Congratulations!** You've applied classification to a real business problem and designed a data-driven retention strategy.

### 🎯 What You Did

- Built a Logistic Regression classifier and learned why a narrow accuracy improvement can still represent genuine business value
- Used **confusion matrices** to understand false positives (wasted spend) vs false negatives (missed revenue)
- Calculated **precision, recall, and accuracy** with sklearn
- Segmented customers by **repurchase probability** to target the "persuadable" group (20-50%)
- Designed a **production workflow** for automated retention campaigns

#### 🏆 Key Takeaways

**Metrics matter:**
- Accuracy alone is misleading for imbalanced data
- Choose **precision** when marketing budget is tight (avoid waste)
- Choose **recall** when intervention is cheap (catch everyone)

**Probabilities > Binary predictions:**
- Not all customers need the same effort
- Target the 20-50% probability group for max ROI
- High probability customers will return anyway

**Models only matter when deployed:**
- Integrate predictions into CRM workflows
- A/B test to prove ROI
- Retrain regularly as customer behavior changes

**Well done!** 💪